In [ ]:
%load_ext autoreload
%autoreload 2
# %matplotlib widget
%pdb off

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, HTML
import pyafn
from pyafn import rho, Cd
import random
from scipy.stats import lognorm
from scipy.stats import norm
from scipy.optimize import curve_fit
from emulationHelpers import readEmulationMI, plot_ventilation_model_fit
from flowEmulationUtils import aggregate_window_series_to_room

#close all figures
plt.close('all')
plt.rcParams['figure.dpi'] = 140
im_scaling = .75
plt.rcParams['figure.figsize'] = [6.4 * im_scaling, 4.8 * im_scaling]
plt.rcParams['font.family'] = ['STIXGeneral', 'DejaVu Serif', 'serif']

home_dir = "./"
display(home_dir)
plt.close('all')

# Load Data

In [ ]:
flowStatsMI, roomVentilationMI = readEmulationMI(home_dir=home_dir)
NWindows = len(flowStatsMI[flowStatsMI['openingType'] == 'xwindow'])
NWindows += len(flowStatsMI[flowStatsMI['openingType'] == 'zwindow'])
display(f"N Windows: {NWindows}")
display(f"N Skylight: {len(flowStatsMI[flowStatsMI['openingType'] == 'skylight'])}")
display(f"N Rooms: {len(roomVentilationMI)}")

In [ ]:
y_var = "flux-Norm"
x_var = "p-noInt_optp0-q_model-Norm"
df = flowStatsMI.copy()
data = df.copy()
room_type_order = ["cross", "corner", "dual", "single"]


In [ ]:
# Use the plotting function with turbulence model
fig, axs, metrics_ps_baseline = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="Pseudo-Steady Baseline",
    p0=[1.0, 0.00000001],
    bounds=([0.99999, 0], [1.00001, 0.0000001]),
    adjustData=False,
    show_assymptotes=True,
    return_metrics=True,
)


In [ ]:
# Use the plotting function with turbulence model
fig, axs, metrics_ps_fit = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="Pseudo-Steady Fit PN",
    p0=[1.0, 0.00000001],
    bounds=([0.1, 0], [np.inf, 0.0000001]),
    adjustData=False,
    show_assymptotes=True,
    return_metrics=True,
)


In [ ]:
# Use the plotting function with turbulence model
fig, axs, metrics_iq_fit = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ Bulk Fit PN",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True,
    return_metrics=True,
)


In [ ]:
x_var2 = "rms-mass_flux-Norm"
q_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2].mean().to_dict()
q_rms_global = data[x_var2].mean()
print(f"Global mean {x_var2}: {q_rms_global:.4f}")
for sl in [0, 1]:
    for s in [1, -1]:
        label = "Skylight" if sl == 1 else "Window"
        direction = "In" if s == 1 else "Out"
        print(f"{label}-{direction} mean {x_var2}: {q_rms_means.get((sl, s), np.nan):.4f}")

# Use subgroup-specific initialization for the second fit parameter
fig, axs, metrics_iq_bulk = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ Bulk Measured PN",
    p0=[1.0, q_rms_global],
    bounds=([0.1, q_rms_global-0.00001], [np.inf, q_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=q_rms_means,
    group_param2_half_width=0.00001,
    return_metrics=True,
 )


In [ ]:
def ventilationReDecomp_qMeasured(X, a, u_rms):
    u_model, rms_scaling = X
    u_rms *= rms_scaling
    return pyafn.ventilationReDecomp_q(u_model, a, u_rms)

plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=x_var2,
    hue=x_var2,
    style="slAll",
    model_func=ventilationReDecomp_qMeasured,
    model_name="Turbulence-Enhanced q",
    p0=[1.0, 1.0],
    bounds=([0.1, .9999], [np.inf, 1.00001]),
    adjustData=False,
    show_assymptotes=False,
    add_numeric_colorbar=True
 )

In [ ]:
# Use the plotting function with turbulence model
fig, axs, metrics_ip_fit = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="$\sigma_{\Delta C_p}$ Bulk Fit PN",
    p0=[1.0, 0.01],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True,
    return_metrics=True,
)


In [ ]:
x_var2 = "p_rms-noInt-Norm"
p_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2].mean().to_dict()
p_rms_global = data[x_var2].mean()
print(f"Global mean {x_var2}: {p_rms_global:.4f}")
for sl in [0, 1]:
    for s in [1, -1]:
        label = "Skylight" if sl == 1 else "Window"
        direction = "In" if s == 1 else "Out"
        print(f"{label}-{direction} mean {x_var2}: {p_rms_means.get((sl, s), np.nan):.4f}")

# Use subgroup-specific initialization for the second fit parameter
fig, axs, metrics_ip_bulk = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="$\sigma_{\Delta C_p}$ Bulk Measured PN",
    p0=[1.0, p_rms_global],
    bounds=([0.1, p_rms_global-0.00001], [np.inf, p_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=p_rms_means,
    group_param2_half_width=0.00001,
    return_metrics=True,
 )


In [ ]:
def ventilationReDecomp_pMeasured(X, a, p_rms):
    u_model, rms_scaling = X
    p_rms *= rms_scaling
    return pyafn.ventilationReDecomp_p(u_model, a, p_rms)

# Combined paper-ready q+p figure (rows: Window/Skylight; cols: q fit, q measured, p fit, p measured)
fig_p, axs_p = plt.subplots(2, 4, figsize=(22, 11), dpi=160, sharex=True, sharey=True)

# Column 1: baseline q model (categorical hue)
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ Bulk Fit PN",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True,
    axes=axs_p[:, 0],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Column 2: subgroup-mean initialized q model (categorical hue)
x_var2_q = "rms-mass_flux-Norm"
q_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2_q].mean().to_dict()
q_rms_global = data[x_var2_q].mean()
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ Bulk Measured PN",
    p0=[1.0, q_rms_global],
    bounds=([0.1, q_rms_global-0.00001], [np.inf, q_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=q_rms_means,
    group_param2_half_width=0.00001,
    axes=axs_p[:, 1],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Column 3: baseline p model (categorical hue)
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="$\sigma_{\Delta C_p}$ Bulk Fit PN",
    p0=[1.0, 0.01],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True,
    axes=axs_p[:, 2],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Column 4: subgroup-mean initialized p model (categorical hue)
x_var2_p = "p_rms-noInt-Norm"
p_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2_p].mean().to_dict()
p_rms_global = data[x_var2_p].mean()
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="$\sigma_{\Delta C_p}$ Bulk Measured PN",
    p0=[1.0, p_rms_global],
    bounds=([0.1, p_rms_global-0.00001], [np.inf, p_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=p_rms_means,
    group_param2_half_width=0.00001,
    axes=axs_p[:, 3],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Paper-ready labels and layout
row_labels_p = ["Windows", "Skylights"]
for i, row_label in enumerate(row_labels_p):
    axs_p[i, 0].set_title(row_label, fontsize=20, loc="left", pad=10)

model_labels_p = [
    "$\sigma_{q_n}$ Bulk Fit PN",
    "$\sigma_{q_n}$ Bulk Measured PN",
    "$\sigma_{\Delta C_p}$ Bulk Fit PN",
    "$\sigma_{\Delta C_p}$ Bulk Measured PN",
]

for ax in axs_p.flatten():
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.tick_params(labelsize=16)
for j, model_label in enumerate(model_labels_p):
    axs_p[1, j].set_xlabel(model_label, fontsize=18)

fig_p.supylabel("LES Flux Velocity", fontsize=24, x=0.04)
# fig_p.suptitle("Ventilation Model Fits (q and p): Window vs Skylight", fontsize=24)
fig_p.subplots_adjust(left=0.10, right=0.98, top=0.94, bottom=0.16, wspace=0.16, hspace=0.18)



In [ ]:
data["Cp"] = data["p_rms-noInt-Norm"] / (rho / 2)

# Combined point-wise figure (rows: Window/Skylight; cols: q and p)
fig_pointwise, axs_pointwise = plt.subplots(2, 2, figsize=(13, 12), dpi=160, sharex=True, sharey=True)

# Point-wise q (left column)
q_hue_norm = plt.Normalize(vmin=0, vmax=0.25)
data_q_pointwise = data.copy()
_, _, metrics_iq_pointwise = plot_ventilation_model_fit(
    data=data_q_pointwise,
    y_var=y_var,
    x_var=x_var,
    x_var2=x_var2_q,
    hue=x_var2_q,
    style="slAll",
    model_func=ventilationReDecomp_qMeasured,
    model_name="$\sigma_{q_n}$ Pointwise Measured PN",
    p0=[1.0, 1.0],
    bounds=([0.1, .9999], [np.inf, 1.00001]),
    adjustData=True,
    show_assymptotes=True,
    axes=axs_pointwise[:, 0],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=True,
    hue_norm=q_hue_norm,
    palette_numeric="plasma",
    figure_suptitle=False,
    colorbar_rect=[0.16, 0.15, 0.30, 0.018],
    colorbar_orientation="horizontal",
    colorbar_label=r"$\sigma_{q_n}$",
    return_metrics=True,
)

# Point-wise p (right column)
x_var2_p = "p_rms-noInt-Norm"
p_hue_norm = plt.Normalize(vmin=0, vmax=0.25)

data_p_pointwise = data.copy()
_, _, metrics_ip_pointwise = plot_ventilation_model_fit(
    data=data_p_pointwise,
    y_var=y_var,
    x_var=x_var,
    x_var2=x_var2_p,
    hue="Cp",
    style="slAll",
    model_func=ventilationReDecomp_pMeasured,
    model_name="$\sigma_{\Delta C_p}$ Pointwise Measured PN",
    p0=[1.0, 1.0],
    bounds=([0.1, .9999], [np.inf, 1.00001]),
    adjustData=True,
    show_assymptotes=False,
    axes=axs_pointwise[:, 1],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=True,
    hue_norm=p_hue_norm,
    palette_numeric="viridis",
    figure_suptitle=False,
    colorbar_rect=[0.61, 0.15, 0.30, 0.018],
    colorbar_orientation="horizontal",
    colorbar_label=r"$\sigma_{\Delta C_p}$",
    return_metrics=True,
)

# Labels and layout
row_labels_pointwise = ["Windows", "Skylights"]
for i, row_label in enumerate(row_labels_pointwise):
    axs_pointwise[i, 0].set_title(row_label, fontsize=20, loc="left", pad=10)

model_labels_pointwise = [
    "$\sigma_{q_n}$ Pointwise Measured PN",
    "$\sigma_{\Delta C_p}$ Pointwise Measured PN",
]

for ax in axs_pointwise.flatten():
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.tick_params(labelsize=16)
for j, model_label in enumerate(model_labels_pointwise):
    axs_pointwise[1, j].set_xlabel(model_label, fontsize=18)

fig_pointwise.supylabel("LES Flux Velocity", fontsize=24, x=0.04)
# fig_pointwise.suptitle("Pointwise Ventilation Model Fits", fontsize=24)
fig_pointwise.subplots_adjust(left=0.12, right=0.94, top=0.90, bottom=0.24, wspace=0.2, hspace=0.22)



In [ ]:
# Row 1: baseline q model (categorical hue)
fig, ax, xAdjusted, fittedParams = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ Bulk Fit PN",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=True,
    show_assymptotes=True,
    add_numeric_colorbar=False,
    return_data=True,
    return_params=True
)

In [ ]:
plt.figure()
plt.scatter(xAdjusted, data[y_var], alpha=0.5, s=1)

In [ ]:
minVent = roomVentilationMI[x_var].min()
maxVent = roomVentilationMI[x_var].max()
x_space = np.linspace(minVent, maxVent, 100)

windowModels = {}
for key, params in fittedParams.items():
    windowModels[key] = pyafn.ventilationReDecomp_q(x_space, *params)

roomModels = {}
roomResiduals = {}
for inOpening in ["Window", "Skylight"]:
    for outOpening in ["Window", "Skylight"]:
        roomModels[f"In {inOpening}; Out {outOpening}"] = (windowModels[(inOpening, 'Flow Entering')] + windowModels[(outOpening, 'Flow Exiting')]) / 2
        roomResiduals[f"In {inOpening}; Out {outOpening}"] = windowModels[(inOpening, 'Flow Entering')] - windowModels[(outOpening, 'Flow Exiting')]


In [ ]:
roomVentilationMI_adj = aggregate_window_series_to_room(
    roomVentilationMI, xAdjusted, out_col="xAdjusted_room", mode="abs_half_sum"
)

roomVentilationMI_adj = aggregate_window_series_to_room(
    roomVentilationMI_adj, xAdjusted, out_col="xAdjusted_room_residual", mode="mean"
)
roomVentilationMI_adj["xAdjusted_room_residual"] *= 2
roomVentilationMI_adj["xAdjusted_diff"] = roomVentilationMI_adj["xAdjusted_room"] - roomVentilationMI_adj[x_var]

plot_df = roomVentilationMI_adj.copy().sort_values("roomType")

fig, axs = plt.subplots(2, 2, figsize=(13, 11), dpi=160)

for i in range(2):
    ax = axs[0, i]
    ax.plot(x_space, x_space, 'r--', label='1:1 Line')

styles = ['-', '--', '-.', ':']
for i, (key, model) in enumerate(roomModels.items()):
    color = "k"#plt.get_cmap("tab20")(i % 20)
    axs[1, 0].plot(x_space, model, linestyle=styles[i % len(styles)], label=key, color=color, alpha=1, linewidth = 1)
    axs[1, 0].legend(fontsize=16)

for i, (key, residual) in enumerate(roomResiduals.items()):
    color = "k" #plt.get_cmap("tab20")(i % 20)
    axs[1, 1].plot(
        roomModels[key] - x_space,
        residual,
        linestyle=styles[i % len(styles)],
        label=key,
        color=color,
        alpha=1,
        linewidth = 1
    )
    axs[1, 1].set_xlim(-0.1, 0.1)
    axs[1, 1].set_ylim(-0.1, 0.1)

scatter_specs = [
    (x_var, "flux-Norm", "Baseline Model vs LES"),
    ("xAdjusted_room", "flux-Norm", "Aggregated Adjusted Model vs LES"),
    (x_var, "xAdjusted_room", "Baseline Model vs Aggregated Adjusted Model"),
    ("xAdjusted_diff", "xAdjusted_room_residual", "Aggregation Difference vs Residual"),
]

for ax, (xcol, ycol, title) in zip(axs.flatten(), scatter_specs):
    sns.scatterplot(
        data=plot_df,
        x=xcol,
        y=ycol,
        hue="roomType",
        style = "slAll",
        palette="colorblind",
        hue_order=room_type_order,
        alpha=0.6,
        s=10,
        ax=ax,
        legend=ax is axs[0, 0],
    )
 
    ax.set_title(title, fontsize=20)
    ax.set_xlabel(xcol, fontsize=18)
    ax.set_ylabel(ycol, fontsize=18)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=16)

handles, labels = axs[0, 0].get_legend_handles_labels()

for ax in axs.flatten():
    if ax.get_legend() is not None:
        ax.get_legend().remove()

fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(0.90, 0.5),
    fontsize=18,
    title="Room Type",
    title_fontsize=18,
    frameon=False,
)

fig.suptitle("Room-Level Aggregated Ventilation Comparisons", fontsize=24)
fig.subplots_adjust(left=0.10, right=0.87, top=0.90, bottom=0.10, wspace=0.28, hspace=0.30)

In [ ]:
def _params_for_opening(opening_type):
    group = "Skylight" if "skylight" in str(opening_type).lower() else "Window"
    return (
        [fittedParams[(group, "Flow Entering")][0]*Cd, fittedParams[(group, "Flow Exiting")][0]*Cd],
        [fittedParams[(group, "Flow Entering")][1], fittedParams[(group, "Flow Exiting")][1]],
    )

def _direct_network_param_lookup_from_fitted_params(fitted_params, measured_sigma=False):
    lookup = {}
    for (opening_group, direction), params in fitted_params.items():
        lookup[(opening_group, direction)] = {
            "cd": params[0] * Cd,
            "sigma": np.nan if measured_sigma else (params[1] if len(params) > 1 else np.nan),
        }
    return lookup

flowStatsMI[["C_d", "pRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

direct_network_param_lookup = _direct_network_param_lookup_from_fitted_params(fittedParams)

flowStatsMI[["C_d", "pRMS"]]

In [ ]:
import flowEmulationUtils as feUtils
flowStatsMI_directNetwork, roomVentilationMI_directNetwork = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [0.6, 0.6], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
def _comparison_metrics(frame, x_spec, y_col):
    if isinstance(x_spec, str):
        x = frame[x_spec]
        x_label = x_spec
    else:
        x = pd.Series(x_spec).reindex(frame.index)
        x_label = getattr(x_spec, "name", None) or "x_adjusted"

    valid = pd.DataFrame({"x": x, "y": frame[y_col]}).dropna()
    residual = valid["x"] - valid["y"]
    rmse = np.sqrt(np.mean(residual ** 2))
    y_std = np.std(valid["y"])
    iqr_error = residual.quantile(0.75) - residual.quantile(0.25)

    return {
        "x_label": x_label,
        "rmse": rmse,
        "nrmse": rmse / y_std if y_std > 0 else np.nan,
        "bias": residual.mean(),
        "iqr_error": iqr_error,
        "std": residual.std(ddof=0),
        "n": len(valid),
    }


def _comparison_row(frame, x_spec, y_col, panel, split):
    metrics = _comparison_metrics(frame, x_spec, y_col)
    if metrics["n"] == 0:
        return None

    return {
        "panel": panel,
        "split": split,
        "rmse": metrics["rmse"],
        "nrmse": metrics["nrmse"],
        "bias": metrics["bias"],
        "iqr_error": metrics["iqr_error"],
        "std": metrics["std"],
        "n": metrics["n"],
        "x_label": metrics["x_label"],
    }


def _flow_subgroup_masks(frame):
    if "Sdelp" not in frame.columns:
        return []

    if "openingType" in frame.columns:
        opening_group = pd.Series(
            np.where(
                frame["openingType"].astype(str).str.contains("skylight", case=False, na=False),
                "Skylight",
                "Window",
            ),
            index=frame.index,
        )
    elif "skylight" in frame.columns:
        opening_group = pd.Series(
            np.where(frame["skylight"].fillna(0).astype(int) == 1, "Skylight", "Window"),
            index=frame.index,
        )
    else:
        return []

    direction_group = pd.Series(None, index=frame.index, dtype="object")
    direction_group.loc[frame["Sdelp"] > 0] = "Flow Entering"
    direction_group.loc[frame["Sdelp"] < 0] = "Flow Exiting"

    masks = []
    for opening_label in ["Window", "Skylight"]:
        for direction_label in ["Flow Entering", "Flow Exiting"]:
            mask = (opening_group == opening_label) & (direction_group == direction_label)
            if mask.any():
                masks.append((f"{opening_label}, {direction_label}", mask))

    return masks


def _panel_plot_data(frame, x_spec):
    plot_frame = frame.copy()
    if isinstance(x_spec, str):
        return plot_frame, x_spec

    x_label = getattr(x_spec, "name", None) or "x_adjusted"
    plot_frame[x_label] = pd.Series(x_spec).reindex(plot_frame.index)
    return plot_frame, x_label


def plot_direct_network_comparison(run_label, flow_frame, room_frame):
    flow_plot = flow_frame.copy()
    room_plot = room_frame.copy()

    flow_plot["p-noInt-direct_optp0-q_model-Norm"] = (
        flow_plot["p-noInt-direct_optp0-q_model"] / flow_plot["WS"]
    )
    room_plot["p-noInt-direct_optp0-q_model-Norm"] = (
        room_plot["p-noInt-direct_optp0-q_model"] / room_plot["WS"]
    )
    flow_plot["flux-Norm"] = flow_plot["flux"] / flow_plot["WS"]
    for measured_col in ["p_rms-noInt-Norm", "rms-mass_flux-Norm"]:
        if measured_col not in flow_plot.columns and measured_col in data.columns:
            flow_plot[measured_col] = data[measured_col].reindex(flow_plot.index)
    if "Cp" not in flow_plot.columns and "p_rms-noInt-Norm" in flow_plot.columns:
        flow_plot["Cp"] = flow_plot["p_rms-noInt-Norm"] / (rho / 2)

    hue_order = room_type_order
    slall_style_col = "Skylight State"
    slall_style_order = ["Skylights Closed", "Skylights Open"]
    measured_opening_hue = {
        "$\sigma_{\Delta C_p}$ Pointwise Measured IN": {
            "hue": "Cp",
            "palette": "viridis",
            "norm": plt.Normalize(vmin=0, vmax=0.25),
            "label": r"$\sigma_{\Delta C_p}$",
        },
        "$\sigma_{q_n}$ Pointwise Measured IN": {
            "hue": "rms-mass_flux-Norm",
            "palette": "plasma",
            "norm": plt.Normalize(vmin=0, vmax=0.25),
            "label": r"$\sigma_{q_n}$",
        },
    }.get(run_label)
    pn_model_label = run_label.replace(" IN", " PN") if run_label.endswith(" IN") else "PN Model"
    in_model_label = run_label if run_label.endswith(" IN") else "IN Model"
    bottom_row_xlabels = ["Pseudo-Steady Baseline", pn_model_label, in_model_label]
    row_titles = ["Windows", "Skylights", "Skylights Closed", "Skylights Open"]

    def _skylight_mask(frame):
        if "openingType" in frame.columns:
            return frame["openingType"].astype(str).str.contains("skylight", case=False, na=False)
        return frame["skylight"].fillna(0).astype(int) == 1

    def _opening_hue_spec(frame):
        if measured_opening_hue is None or measured_opening_hue["hue"] not in frame.columns:
            return {"hue": "roomType", "palette": "colorblind", "hue_order": hue_order}
        return {
            "hue": measured_opening_hue["hue"],
            "palette": measured_opening_hue["palette"],
            "hue_norm": measured_opening_hue["norm"],
            "hue_order": None,
            "numeric_hue": True,
        }

    def _add_slall_style(frame):
        frame = frame.copy()
        if "slAll" in frame.columns:
            frame[slall_style_col] = np.where(frame["slAll"].astype(bool), "Skylights Open", "Skylights Closed")
        else:
            frame[slall_style_col] = "Unknown"
        return frame

    window_data = _add_slall_style(data[~_skylight_mask(data)])
    skylight_data = _add_slall_style(data[_skylight_mask(data)])
    window_flow_plot = _add_slall_style(flow_plot[~_skylight_mask(flow_plot)])
    skylight_flow_plot = _add_slall_style(flow_plot[_skylight_mask(flow_plot)])

    non_slall_room_base = roomVentilationMI[roomVentilationMI["slAll"] == False]
    slall_room_base = roomVentilationMI[roomVentilationMI["slAll"] == True]
    non_slall_room_adj = roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == False]
    slall_room_adj = roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == True]
    non_slall_room_plot = room_plot[room_plot["slAll"] == False]
    slall_room_plot = room_plot[room_plot["slAll"] == True]

    panel_grid = [
        [
            {
                "title": "Window Baseline vs LES",
                "frame": window_data,
                "x": x_var,
                "y": y_var,
                **_opening_hue_spec(window_data),
                "style": slall_style_col,
                "style_order": slall_style_order,
            },
            {
                "title": "Window Adjusted vs LES",
                "frame": window_flow_plot,
                "x": xAdjusted,
                "y": y_var,
                **_opening_hue_spec(window_flow_plot),
                "style": slall_style_col,
                "style_order": slall_style_order,
            },
            {
                "title": f"Window {run_label} vs LES",
                "frame": window_flow_plot,
                "x": "p-noInt-direct_optp0-q_model-Norm",
                "y": y_var,
                **_opening_hue_spec(window_flow_plot),
                "style": slall_style_col,
                "style_order": slall_style_order,
            },
        ],
        [
            {
                "title": "Skylight Baseline vs LES",
                "frame": skylight_data,
                "x": x_var,
                "y": y_var,
                **_opening_hue_spec(skylight_data),
                "style": slall_style_col,
                "style_order": slall_style_order,
            },
            {
                "title": "Skylight Adjusted vs LES",
                "frame": skylight_flow_plot,
                "x": xAdjusted,
                "y": y_var,
                **_opening_hue_spec(skylight_flow_plot),
                "style": slall_style_col,
                "style_order": slall_style_order,
            },
            {
                "title": f"Skylight {run_label} vs LES",
                "frame": skylight_flow_plot,
                "x": "p-noInt-direct_optp0-q_model-Norm",
                "y": y_var,
                **_opening_hue_spec(skylight_flow_plot),
                "style": slall_style_col,
                "style_order": slall_style_order,
            },
        ],
        [
            {
                "title": "Non-slAll Room Baseline vs LES",
                "frame": non_slall_room_base,
                "x": x_var,
                "y": y_var,
                "hue": "roomType",
                "palette": "colorblind",
                "hue_order": hue_order,
                "style": "roomType",
                "style_order": hue_order,
            },
            {
                "title": "Non-slAll Room Adjusted vs LES",
                "frame": non_slall_room_adj,
                "x": "xAdjusted_room",
                "y": y_var,
                "hue": "roomType",
                "palette": "colorblind",
                "hue_order": hue_order,
                "style": "roomType",
                "style_order": hue_order,
            },
            {
                "title": f"Non-slAll Room {run_label} vs LES",
                "frame": non_slall_room_plot,
                "x": "p-noInt-direct_optp0-q_model-Norm",
                "y": y_var,
                "hue": "roomType",
                "palette": "colorblind",
                "hue_order": hue_order,
                "style": "roomType",
                "style_order": hue_order,
            },
        ],
        [
            {
                "title": "slAll Room Baseline vs LES",
                "frame": slall_room_base,
                "x": x_var,
                "y": y_var,
                "hue": "roomType",
                "palette": "colorblind",
                "hue_order": hue_order,
                "style": "roomType",
                "style_order": hue_order,
            },
            {
                "title": "slAll Room Adjusted vs LES",
                "frame": slall_room_adj,
                "x": "xAdjusted_room",
                "y": y_var,
                "hue": "roomType",
                "palette": "colorblind",
                "hue_order": hue_order,
                "style": "roomType",
                "style_order": hue_order,
            },
            {
                "title": f"slAll Room {run_label} vs LES",
                "frame": slall_room_plot,
                "x": "p-noInt-direct_optp0-q_model-Norm",
                "y": y_var,
                "hue": "roomType",
                "palette": "colorblind",
                "hue_order": hue_order,
                "style": "roomType",
                "style_order": hue_order,
            },
        ],
    ]

    fig, axs = plt.subplots(4, 3, figsize=(14, 15), dpi=160)
    stats_rows = []
    legend_ax = axs[2, 0] if measured_opening_hue is not None else axs[0, 0]

    flat_specs = [spec for row in panel_grid for spec in row]
    for panel_idx, (ax, spec) in enumerate(zip(axs.flatten(), flat_specs)):
        row_idx, col_idx = divmod(panel_idx, len(bottom_row_xlabels))
        plot_frame, x_col = _panel_plot_data(spec["frame"], spec["x"])
        scatter_kwargs = {
            "data": plot_frame,
            "x": x_col,
            "y": spec["y"],
            "hue": spec["hue"],
            "style": spec["style"],
            "palette": spec["palette"],
            "ax": ax,
            "alpha": 0.7,
            "s": 20,
            "hue_order": spec.get("hue_order"),
            "style_order": spec["style_order"],
            "legend": False if spec.get("numeric_hue", False) else ax is legend_ax,
        }
        if "hue_norm" in spec:
            scatter_kwargs["hue_norm"] = spec["hue_norm"]
        sns.scatterplot(**scatter_kwargs)

        metrics = _comparison_metrics(spec["frame"], spec["x"], spec["y"])
        total_row = _comparison_row(spec["frame"], spec["x"], spec["y"], spec["title"], "Total")
        if total_row is not None:
            stats_rows.append(total_row)

        for split_label, mask in _flow_subgroup_masks(spec["frame"]):
            split_row = _comparison_row(
                spec["frame"].loc[mask],
                spec["x"],
                spec["y"],
                spec["title"],
                split_label,
            )
            if split_row is not None:
                stats_rows.append(split_row)

        valid = plot_frame[[x_col, spec["y"]]].dropna()
        if not valid.empty:
            lim_min = valid.min().min()
            lim_max = valid.max().max()
            ax.plot([lim_min, lim_max], [lim_min, lim_max], "r--", linewidth=1.0, alpha=0.7)
            ax.set_xlim(lim_min, lim_max)
            ax.set_ylim(lim_min, lim_max)

        ax.set_title(row_titles[row_idx] if col_idx == 0 else "", fontsize=16, loc="left")
        ax.set_xlabel(bottom_row_xlabels[col_idx] if row_idx == len(panel_grid) - 1 else "", fontsize=16)
        ax.set_ylabel("LES Flux Velocity" if col_idx == 0 else "", fontsize=16)
        ax.tick_params(axis="both", labelsize=14)
        ax.grid(True, alpha=0.3)

    handles, labels = axs[2, 0].get_legend_handles_labels()
    if not handles:
        handles, labels = axs[0, 0].get_legend_handles_labels()
    for ax in axs.flatten():
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    if handles:
        fig.legend(
            handles,
            labels,
            loc="center left",
            bbox_to_anchor=(0.92, 0.5),
            fontsize=14,
            frameon=False,
        )

    if measured_opening_hue is not None:
        mappable = plt.cm.ScalarMappable(
            norm=measured_opening_hue["norm"],
            cmap=plt.get_cmap(measured_opening_hue["palette"]),
        )
        mappable.set_array([])
        cbar = fig.colorbar(mappable, ax=axs[:2, :], location="right", fraction=0.025, pad=0.02)
        cbar.set_label(measured_opening_hue["label"], fontsize=16)
        cbar.ax.tick_params(labelsize=14)

    fig.suptitle(f"{run_label} Comparison", fontsize=18)
    fig.subplots_adjust(right=0.88, hspace=0.45, wspace=0.25)

    stats_df = pd.DataFrame(stats_rows)
    print(run_label)
    overall_stats = stats_df[stats_df["split"] == "Total"].drop(columns=["split", "x_label"])
    print("Overall")
    print(overall_stats.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    subgroup_stats = stats_df[stats_df["split"] != "Total"].drop(columns=["x_label"])
    if not subgroup_stats.empty:
        print("\nBy opening / flow direction")
        print(subgroup_stats.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    return fig, axs, stats_df


fig, axs, stats_direct_network = plot_direct_network_comparison(
    "Direct Network",
    flowStatsMI_directNetwork,
    roomVentilationMI_directNetwork,
)


In [ ]:
optParams =  [6.647e-01, 6.558e-01, 5.395e-01, 6.928e-01, 2.923e-01, 1.106e-01, -1.354e-08, 1.775e-01]

fittedParams = {
    ('Window',   'Flow Entering'): np.array([optParams[0]/Cd, optParams[4]]),
    ('Window',   'Flow Exiting'):  np.array([optParams[1]/Cd, optParams[5]]),
    ('Skylight', 'Flow Entering'): np.array([optParams[2]/Cd, optParams[6]]),
    ('Skylight', 'Flow Exiting'):  np.array([optParams[3]/Cd, optParams[7]])
    }

def _params_for_opening(opening_type):
    group = "Skylight" if "skylight" in str(opening_type).lower() else "Window"
    return (
        [fittedParams[(group, "Flow Entering")][0]*Cd, fittedParams[(group, "Flow Exiting")][0]*Cd],
        [fittedParams[(group, "Flow Entering")][1], fittedParams[(group, "Flow Exiting")][1]],
    )

flowStatsMI[["C_d", "qRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

direct_network1_param_lookup = _direct_network_param_lookup_from_fitted_params(fittedParams)

flowStatsMI[["C_d", "qRMS"]]

In [ ]:
flowStatsMI.drop(columns=["pRMS"], inplace=True)
flowStatsMI_directNetwork1, roomVentilationMI_directNetwork1 = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [0.6, 0.6], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
fig, axs, stats_direct_network1 = plot_direct_network_comparison(
    "$\sigma_{q_n}$ Bulk Fit IN",
    flowStatsMI_directNetwork1,
    roomVentilationMI_directNetwork1,
)

In [ ]:
optParams = [6.121e-01, 6.467e-01, 5.840e-01, 6.464e-01, 1.271e-01, 1.010e-01, 9.888e-09, 1.309e-01]

fittedParams = {
    ('Window',   'Flow Entering'): np.array([optParams[0]/Cd, optParams[4]]),
    ('Window',   'Flow Exiting'):  np.array([optParams[1]/Cd, optParams[5]]),
    ('Skylight', 'Flow Entering'): np.array([optParams[2]/Cd, optParams[6]]),
    ('Skylight', 'Flow Exiting'):  np.array([optParams[3]/Cd, optParams[7]])
    }

def _params_for_opening(opening_type):
    group = "Skylight" if "skylight" in str(opening_type).lower() else "Window"
    return (
        [fittedParams[(group, "Flow Entering")][0]*Cd, fittedParams[(group, "Flow Exiting")][0]*Cd],
        [fittedParams[(group, "Flow Entering")][1], fittedParams[(group, "Flow Exiting")][1]],
    )

flowStatsMI[["C_d", "pRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

direct_network2_param_lookup = _direct_network_param_lookup_from_fitted_params(fittedParams)

flowStatsMI[["C_d", "pRMS"]]

In [ ]:
flowStatsMI.drop(columns=["qRMS"], inplace=True)
flowStatsMI_directNetwork2, roomVentilationMI_directNetwork2 = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [0.6, 0.6], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
fig, axs, stats_direct_network2 = plot_direct_network_comparison(
    "$\sigma_{\Delta C_p}$ Bulk Fit IN",
    flowStatsMI_directNetwork2,
    roomVentilationMI_directNetwork2,
)


In [ ]:
def _skylight_mask(frame):
    if "openingType" in frame.columns:
        return frame["openingType"].astype(str).str.contains("skylight", case=False, na=False)
    return frame["skylight"].fillna(0).astype(int) == 1

flow_direct_plot = flowStatsMI_directNetwork.copy()
flow_direct_plot["p-noInt-direct_optp0-q_model-Norm"] = (
    flow_direct_plot["p-noInt-direct_optp0-q_model"] / flow_direct_plot["WS"]
)
flow_direct_plot["flux-Norm"] = flow_direct_plot["flux"] / flow_direct_plot["WS"]

room_direct_plot = roomVentilationMI_directNetwork.copy()
room_direct_plot["p-noInt-direct_optp0-q_model-Norm"] = (
    room_direct_plot["p-noInt-direct_optp0-q_model"] / room_direct_plot["WS"]
)

flow_direct2_plot = flowStatsMI_directNetwork2.copy()
flow_direct2_plot["p-noInt-direct_optp0-q_model-Norm"] = (
    flow_direct2_plot["p-noInt-direct_optp0-q_model"] / flow_direct2_plot["WS"]
)
flow_direct2_plot["flux-Norm"] = flow_direct2_plot["flux"] / flow_direct2_plot["WS"]

room_direct2_plot = roomVentilationMI_directNetwork2.copy()
room_direct2_plot["p-noInt-direct_optp0-q_model-Norm"] = (
    room_direct2_plot["p-noInt-direct_optp0-q_model"] / room_direct2_plot["WS"]
)

slall_style_col = "Skylight State"
slall_style_order = ["Skylights Closed", "Skylights Open"]

def _add_slall_style(frame):
    frame = frame.copy()
    if "slAll" in frame.columns:
        frame[slall_style_col] = np.where(frame["slAll"].astype(bool), "Skylights Open", "Skylights Closed")
    else:
        frame[slall_style_col] = "Unknown"
    return frame

window_data = _add_slall_style(data[~_skylight_mask(data)])
skylight_data = _add_slall_style(data[_skylight_mask(data)])
window_direct_plot = _add_slall_style(flow_direct_plot[~_skylight_mask(flow_direct_plot)])
skylight_direct_plot = _add_slall_style(flow_direct_plot[_skylight_mask(flow_direct_plot)])
window_direct2_plot = _add_slall_style(flow_direct2_plot[~_skylight_mask(flow_direct2_plot)])
skylight_direct2_plot = _add_slall_style(flow_direct2_plot[_skylight_mask(flow_direct2_plot)])

non_slall_room_base = roomVentilationMI[roomVentilationMI["slAll"] == False]
slall_room_base = roomVentilationMI[roomVentilationMI["slAll"] == True]
non_slall_room_adj = roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == False]
slall_room_adj = roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == True]
non_slall_room_direct = room_direct_plot[room_direct_plot["slAll"] == False]
slall_room_direct = room_direct_plot[room_direct_plot["slAll"] == True]
non_slall_room_direct2 = room_direct2_plot[room_direct2_plot["slAll"] == False]
slall_room_direct2 = room_direct2_plot[room_direct2_plot["slAll"] == True]

def _opening_panel(title, frame, x_col):
    return {
        "title": title,
        "frame": frame,
        "x": x_col,
        "y": y_var,
        "hue": "roomType",
        "palette": "colorblind",
        "hue_order": room_type_order,
        "style": slall_style_col,
        "style_order": slall_style_order,
    }

def _room_panel(title, frame, x_col):
    return {
        "title": title,
        "frame": frame,
        "x": x_col,
        "y": y_var,
        "hue": "roomType",
        "palette": "colorblind",
        "hue_order": room_type_order,
        "style": "roomType",
        "style_order": room_type_order,
    }

panel_grid = [
    [
        _opening_panel("Window Baseline vs LES", window_data, x_var),
        _opening_panel("Window Adjusted vs LES", window_direct2_plot, xAdjusted),
        _opening_panel("Window Legacy Direct Network vs LES", window_direct_plot, "p-noInt-direct_optp0-q_model-Norm"),
        _opening_panel("Window $\sigma_{\Delta C_p}$ Bulk Fit IN vs LES", window_direct2_plot, "p-noInt-direct_optp0-q_model-Norm"),
    ],
    [
        _opening_panel("Skylight Baseline vs LES", skylight_data, x_var),
        _opening_panel("Skylight Adjusted vs LES", skylight_direct2_plot, xAdjusted),
        _opening_panel("Skylight Legacy Direct Network vs LES", skylight_direct_plot, "p-noInt-direct_optp0-q_model-Norm"),
        _opening_panel("Skylight $\sigma_{\Delta C_p}$ Bulk Fit IN vs LES", skylight_direct2_plot, "p-noInt-direct_optp0-q_model-Norm"),
    ],
    [
        _room_panel("Non-slAll Room Baseline vs LES", non_slall_room_base, x_var),
        _room_panel("Non-slAll Room Adjusted vs LES", non_slall_room_adj, "xAdjusted_room"),
        _room_panel("Non-slAll Room Legacy Direct Network vs LES", non_slall_room_direct, "p-noInt-direct_optp0-q_model-Norm"),
        _room_panel("Non-slAll Room $\sigma_{\Delta C_p}$ Bulk Fit IN vs LES", non_slall_room_direct2, "p-noInt-direct_optp0-q_model-Norm"),
    ],
    [
        _room_panel("slAll Room Baseline vs LES", slall_room_base, x_var),
        _room_panel("slAll Room Adjusted vs LES", slall_room_adj, "xAdjusted_room"),
        _room_panel("slAll Room Legacy Direct Network vs LES", slall_room_direct, "p-noInt-direct_optp0-q_model-Norm"),
        _room_panel("slAll Room $\sigma_{\Delta C_p}$ Bulk Fit IN vs LES", slall_room_direct2, "p-noInt-direct_optp0-q_model-Norm"),
    ],
]

fig, axs = plt.subplots(4, 4, figsize=(18, 15), dpi=160)

for ax, spec in zip(axs.flatten(), [spec for row in panel_grid for spec in row]):
    plot_frame, x_col = _panel_plot_data(spec["frame"], spec["x"])
    sns.scatterplot(
        data=plot_frame,
        x=x_col,
        y=spec["y"],
        hue=spec["hue"],
        style=spec["style"],
        palette=spec["palette"],
        hue_order=spec["hue_order"],
        style_order=spec["style_order"],
        ax=ax,
        alpha=0.7,
        s=20,
        legend=ax is axs[0, 0],
    )

    metrics = _comparison_metrics(spec["frame"], spec["x"], spec["y"])
    valid = plot_frame[[x_col, spec["y"]]].dropna()
    if not valid.empty:
        lim_min = valid.min().min()
        lim_max = valid.max().max()
        ax.plot([lim_min, lim_max], [lim_min, lim_max], "r--", linewidth=1.0, alpha=0.7)
        ax.set_xlim(lim_min, lim_max)
        ax.set_ylim(lim_min, lim_max)

    ax.set_title(spec["title"], fontsize=12)
    ax.set_xlabel(metrics["x_label"], fontsize=10)
    ax.set_ylabel(spec["y"], fontsize=10)
    ax.grid(True, alpha=0.3)

handles, labels = axs[0, 0].get_legend_handles_labels()
for ax in axs.flatten():
    if ax.get_legend() is not None:
        ax.get_legend().remove()

if handles:
    fig.legend(
        handles,
        labels,
        loc="center left",
        bbox_to_anchor=(0.92, 0.5),
        frameon=False,
    )

fig.suptitle("Legacy Direct Network vs $\sigma_{\Delta C_p}$ Bulk Fit IN Comparison", fontsize=18)
fig.subplots_adjust(right=0.88, hspace=0.45, wspace=0.3)


In [ ]:
optParams = [6.882e-01, 7.107e-01, 6.848e-01, 7.639e-01]

fittedParams = {
    ('Window',   'Flow Entering'): np.array([optParams[0]/Cd, 0]),
    ('Window',   'Flow Exiting'):  np.array([optParams[1]/Cd, 0]),
    ('Skylight', 'Flow Entering'): np.array([optParams[2]/Cd, 0]),
    ('Skylight', 'Flow Exiting'):  np.array([optParams[3]/Cd, 0])
    }

flowStatsMI[["C_d", "pRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

direct_network3_param_lookup = _direct_network_param_lookup_from_fitted_params(fittedParams, measured_sigma=True)

flowStatsMI["pRMS"] = flowStatsMI.apply(lambda x: [x["p_rms-noInt"], x["p_rms-noInt"]], axis=1)

flowStatsMI[["C_d", "pRMS"]]

In [ ]:
flowStatsMI_directNetwork3, roomVentilationMI_directNetwork3 = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [0.6, 0.6], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
fig, axs, stats_direct_network3 = plot_direct_network_comparison(
    "$\sigma_{\Delta C_p}$ Pointwise Measured IN",
    flowStatsMI_directNetwork3,
    roomVentilationMI_directNetwork3,
)


In [ ]:
optParams = [7.022e-01, 7.195e-01, 6.934e-01, 7.408e-01]

fittedParams = {
    ('Window',   'Flow Entering'): np.array([optParams[0]/Cd, 0]),
    ('Window',   'Flow Exiting'):  np.array([optParams[1]/Cd, 0]),
    ('Skylight', 'Flow Entering'): np.array([optParams[2]/Cd, 0]),
    ('Skylight', 'Flow Exiting'):  np.array([optParams[3]/Cd, 0])
    }

flowStatsMI[["C_d", "qRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

direct_network4_param_lookup = _direct_network_param_lookup_from_fitted_params(fittedParams, measured_sigma=True)

flowStatsMI["qRMS"] = flowStatsMI.apply(lambda x: [x["rms-mass_flux"], x["rms-mass_flux"]], axis=1)
flowStatsMI[["C_d", "qRMS"]]

In [ ]:
flowStatsMI.drop(columns=["pRMS"], inplace=True)
flowStatsMI_directNetwork4, roomVentilationMI_directNetwork4 = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [0.6, 0.6], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
fig, axs, stats_direct_network4 = plot_direct_network_comparison(
    "$\sigma_{q_n}$ Pointwise Measured IN",
    flowStatsMI_directNetwork4,
    roomVentilationMI_directNetwork4,
)


In [ ]:

def _ensure_iqr_error_column(df):
    if "iqr_error" in df.columns:
        return df
    df = df.copy()
    for candidate in ["iqr", "error_iqr"]:
        if candidate in df.columns:
            df["iqr_error"] = df[candidate]
            return df
    df["iqr_error"] = np.nan
    return df


def _apply_direct_network_params(df, param_lookup):
    if param_lookup is None:
        return df

    df = df.copy()
    for (opening_group, direction), params in param_lookup.items():
        mask = (df["opening_group"] == opening_group) & (df["direction"] == direction)
        df.loc[mask, "cd"] = params["cd"]
        df.loc[mask, "sigma"] = params["sigma"]
    return df


def _standardize_fit_metrics(metrics_df):
    df = _ensure_iqr_error_column(metrics_df)
    df["source"] = "plot_ventilation_model_fit"
    df["run_label"] = df["model_name"]
    return df[
        [
            "source",
            "run_label",
            "model_name",
            "scope",
            "opening_group",
            "direction",
            "split",
            "adjusted",
            "cd",
            "sigma",
            "rmse",
            "nrmse",
            "bias",
            "iqr_error",
            "std",
            "n",
            "x_label",
            "y_label",
        ]
    ]


def _standardize_direct_network_metrics(run_label, stats_df, param_lookup=None):
    df = _ensure_iqr_error_column(stats_df)
    split_parts = df["split"].str.split(",", n=1, expand=True)
    df["source"] = "direct_network"
    df["run_label"] = run_label
    df["model_name"] = df["panel"]
    df["scope"] = np.where(df["split"].eq("Total"), "panel_total", "subgroup")
    df["opening_group"] = np.where(df["split"].eq("Total"), pd.NA, split_parts[0])
    df["direction"] = np.where(df["split"].eq("Total"), "All", split_parts[1].str.strip())
    df["adjusted"] = df["panel"].str.contains("Adjusted", na=False)
    df["cd"] = np.nan
    df["sigma"] = np.nan
    df = _apply_direct_network_params(df, param_lookup)
    df["y_label"] = y_var
    return df[
        [
            "source",
            "run_label",
            "model_name",
            "scope",
            "opening_group",
            "direction",
            "split",
            "adjusted",
            "cd",
            "sigma",
            "rmse",
            "nrmse",
            "bias",
            "iqr_error",
            "std",
            "n",
            "x_label",
            "y_label",
        ]
    ]


def _room_slall_metrics(run_label, model_name, frame, x_spec, source, adjusted=False):
    rows = []
    room_type_specs = [("All", pd.Series(True, index=frame.index))]
    if "roomType" in frame.columns:
        for room_type in room_type_order:
            room_type_specs.append((room_type, frame["roomType"] == room_type))

    for sl_all, opening_group in [(False, "Window"), (True, "Skylight")]:
        sl_mask = frame["slAll"] == sl_all
        for room_type, room_type_mask in room_type_specs:
            subset = frame[sl_mask & room_type_mask]
            metrics = _comparison_metrics(subset, x_spec, y_var)
            if metrics["n"] == 0:
                continue
            split_label = f"{opening_group} Rooms" if room_type == "All" else f"{opening_group} {room_type} Rooms"
            rows.append(
                {
                    "source": source,
                    "run_label": run_label,
                    "model_name": model_name,
                    "scope": "room_slall",
                    "opening_group": opening_group,
                    "room_type": room_type,
                    "direction": "All",
                    "split": split_label,
                    "adjusted": adjusted,
                    "cd": np.nan,
                    "sigma": np.nan,
                    "rmse": metrics["rmse"],
                    "nrmse": metrics["nrmse"],
                    "bias": metrics["bias"],
                    "iqr_error": metrics["iqr_error"],
                    "std": metrics["std"],
                    "n": metrics["n"],
                    "x_label": metrics["x_label"],
                    "y_label": y_var,
                }
            )
    return pd.DataFrame(rows)


def _model_spec_for_room_aggregation(model_name):
    if "Pointwise" in model_name and "\sigma_{q_n}" in model_name:
        return ventilationReDecomp_qMeasured, x_var2_q
    if "Pointwise" in model_name and "\sigma_{\Delta C_p}" in model_name:
        return ventilationReDecomp_pMeasured, x_var2_p
    if "\sigma_{\Delta C_p}" in model_name:
        return pyafn.ventilationReDecomp_p, None
    return pyafn.ventilationReDecomp_q, None


def _opening_predictions_from_fit_metrics(metrics_df):
    subgroup_metrics = metrics_df[metrics_df["scope"] == "subgroup"].copy()
    if subgroup_metrics.empty:
        return pd.Series(dtype=float)

    model_name = subgroup_metrics["model_name"].iloc[0]
    model_func, x_var2_col = _model_spec_for_room_aggregation(model_name)
    prediction = pd.Series(np.nan, index=data.index, dtype=float, name=f"{model_name}_opening_prediction")

    for _, metric_row in subgroup_metrics.iterrows():
        sl_val = metric_row["opening_group"] == "Skylight"
        sdelp = 1 if metric_row["direction"] == "Flow Entering" else -1
        mask = (data["skylight"].astype(bool) == sl_val) & (data["Sdelp"] == sdelp)
        if not mask.any():
            continue

        popt = [metric_row["cd"] / Cd, metric_row["sigma"]]
        x_abs = np.abs(data.loc[mask, x_var].values)
        if x_var2_col is None:
            y_pred = model_func(x_abs, *popt)
        else:
            x2_abs = np.abs(data.loc[mask, x_var2_col].values)
            y_pred = model_func((x_abs, x2_abs), *popt)
        prediction.loc[mask] = y_pred * sdelp

    return prediction


def _room_metrics_from_fit_metrics(metrics_df):
    model_name = metrics_df["model_name"].iloc[0]
    opening_prediction = _opening_predictions_from_fit_metrics(metrics_df)
    room_prediction_col = f"{model_name}_room_prediction"
    room_prediction_frame = aggregate_window_series_to_room(
        roomVentilationMI,
        opening_prediction,
        out_col=room_prediction_col,
        mode="abs_half_sum",
    )
    return _room_slall_metrics(
        model_name,
        model_name,
        room_prediction_frame,
        room_prediction_col,
        source="plot_ventilation_model_fit",
        adjusted=bool(metrics_df["adjusted"].any()),
    )


def _direct_network_room_slall_metrics(run_label, room_frame):
    room_plot = room_frame.copy()
    direct_col = "p-noInt-direct_optp0-q_model-Norm"
    room_plot[direct_col] = room_plot["p-noInt-direct_optp0-q_model"] / room_plot["WS"]

    return pd.concat(
        [
            _room_slall_metrics(run_label, "Room Baseline vs LES", roomVentilationMI, x_var, "direct_network"),
            _room_slall_metrics(run_label, "Room Adjusted vs LES", roomVentilationMI_adj, "xAdjusted_room", "direct_network", adjusted=True),
            _room_slall_metrics(run_label, f"Room {run_label} vs LES", room_plot, direct_col, "direct_network"),
        ],
        ignore_index=True,
    )


def _direct_network_i_type_metrics(run_label, i_model_name, stats_df, room_frame, param_lookup=None):
    opening_stats = _standardize_direct_network_metrics(run_label, stats_df, param_lookup)
    opening_stats = opening_stats[
        opening_stats["model_name"].isin([
            f"Window {run_label} vs LES",
            f"Skylight {run_label} vs LES",
        ])
        & (opening_stats["scope"] == "subgroup")
    ].copy()

    room_stats = _direct_network_room_slall_metrics(run_label, room_frame)
    room_stats = room_stats[room_stats["model_name"] == f"Room {run_label} vs LES"].copy()

    direct_stats = pd.concat([opening_stats, room_stats], ignore_index=True)
    direct_stats["source"] = "direct_network_i_type"
    direct_stats["run_label"] = i_model_name
    direct_stats["model_name"] = i_model_name
    direct_stats = _apply_direct_network_params(direct_stats, param_lookup)
    return direct_stats


fit_metric_frames = [
    metrics_ps_baseline,
    metrics_ps_fit,
    metrics_iq_fit,
    metrics_iq_bulk,
    metrics_ip_fit,
    metrics_ip_bulk,
    metrics_iq_pointwise,
    metrics_ip_pointwise,
]

direct_network_param_lookups = {
    "$\sigma_{q_n}$ Bulk Fit IN": globals().get("direct_network1_param_lookup"),
    "$\sigma_{\Delta C_p}$ Bulk Fit IN": globals().get("direct_network2_param_lookup"),
    "$\sigma_{\Delta C_p}$ Pointwise Measured IN": globals().get("direct_network3_param_lookup"),
    "$\sigma_{q_n}$ Pointwise Measured IN": globals().get("direct_network4_param_lookup"),
}

direct_network_metric_frames = [
    _standardize_direct_network_metrics("$\sigma_{q_n}$ Bulk Fit IN", stats_direct_network1, direct_network_param_lookups["$\sigma_{q_n}$ Bulk Fit IN"]),
    _standardize_direct_network_metrics("$\sigma_{\Delta C_p}$ Bulk Fit IN", stats_direct_network2, direct_network_param_lookups["$\sigma_{\Delta C_p}$ Bulk Fit IN"]),
    _standardize_direct_network_metrics("$\sigma_{\Delta C_p}$ Pointwise Measured IN", stats_direct_network3, direct_network_param_lookups["$\sigma_{\Delta C_p}$ Pointwise Measured IN"]),
    _standardize_direct_network_metrics("$\sigma_{q_n}$ Pointwise Measured IN", stats_direct_network4, direct_network_param_lookups["$\sigma_{q_n}$ Pointwise Measured IN"]),
]

fit_room_metric_frames = [_room_metrics_from_fit_metrics(df) for df in fit_metric_frames]

direct_network_room_metric_frames = [
    _direct_network_room_slall_metrics("$\sigma_{q_n}$ Bulk Fit IN", roomVentilationMI_directNetwork1),
    _direct_network_room_slall_metrics("$\sigma_{\Delta C_p}$ Bulk Fit IN", roomVentilationMI_directNetwork2),
    _direct_network_room_slall_metrics("$\sigma_{\Delta C_p}$ Pointwise Measured IN", roomVentilationMI_directNetwork3),
    _direct_network_room_slall_metrics("$\sigma_{q_n}$ Pointwise Measured IN", roomVentilationMI_directNetwork4),
]

direct_network_i_type_metric_frames = [
    _direct_network_i_type_metrics(
        "$\sigma_{q_n}$ Bulk Fit IN",
        "$\sigma_{q_n}$ Bulk Fit IN",
        stats_direct_network1,
        roomVentilationMI_directNetwork1,
        direct_network_param_lookups["$\sigma_{q_n}$ Bulk Fit IN"],
    ),
    _direct_network_i_type_metrics(
        "$\sigma_{q_n}$ Pointwise Measured IN",
        "$\sigma_{q_n}$ Pointwise Measured IN",
        stats_direct_network4,
        roomVentilationMI_directNetwork4,
        direct_network_param_lookups["$\sigma_{q_n}$ Pointwise Measured IN"],
    ),
    _direct_network_i_type_metrics(
        "$\sigma_{\Delta C_p}$ Bulk Fit IN",
        "$\sigma_{\Delta C_p}$ Bulk Fit IN",
        stats_direct_network2,
        roomVentilationMI_directNetwork2,
        direct_network_param_lookups["$\sigma_{\Delta C_p}$ Bulk Fit IN"],
    ),
    _direct_network_i_type_metrics(
        "$\sigma_{\Delta C_p}$ Pointwise Measured IN",
        "$\sigma_{\Delta C_p}$ Pointwise Measured IN",
        stats_direct_network3,
        roomVentilationMI_directNetwork3,
        direct_network_param_lookups["$\sigma_{\Delta C_p}$ Pointwise Measured IN"],
    ),
]

error_metrics_tidy = pd.concat(
    [_standardize_fit_metrics(df) for df in fit_metric_frames]
    + direct_network_metric_frames
    + fit_room_metric_frames
    + direct_network_room_metric_frames
    + direct_network_i_type_metric_frames,
    ignore_index=True,
)

error_metrics_tidy = error_metrics_tidy.sort_values(
    ["source", "run_label", "scope", "opening_group", "direction"]
).reset_index(drop=True)

display(error_metrics_tidy)


In [ ]:
import os

ashrae_baseline_label = "ASHRAE Baseline"
ashrae_q_label = r"ASHRAE $\sigma_{q_n}$ Fit"
ashrae_p_label = r"ASHRAE $\sigma_{\Delta C_p}$ Fit"
ashrae_error_metrics_path = os.path.join(home_dir, "ashrae_exports", "ashrae_error_metrics.csv")
ashrae_fit_params_path = os.path.join(home_dir, "ashrae_exports", "ashrae_fit_params.csv")

original_table_model_order = [
    "Pseudo-Steady Baseline",
    "Pseudo-Steady Fit PN",
    "$\sigma_{q_n}$ Bulk Fit PN",
    "$\sigma_{q_n}$ Bulk Fit IN",
    "$\sigma_{q_n}$ Bulk Measured PN",
    "$\sigma_{q_n}$ Pointwise Measured PN",
    "$\sigma_{q_n}$ Pointwise Measured IN",
    "$\sigma_{\Delta C_p}$ Bulk Fit PN",
    "$\sigma_{\Delta C_p}$ Bulk Fit IN",
    "$\sigma_{\Delta C_p}$ Bulk Measured PN",
    "$\sigma_{\Delta C_p}$ Pointwise Measured PN",
    "$\sigma_{\Delta C_p}$ Pointwise Measured IN",
    ashrae_baseline_label,
    ashrae_q_label,
    ashrae_p_label,
]

original_table_label_map = {}

def _add_iqr_error_column(df):
    if "iqr_error" in df.columns:
        return df
    for candidate in ["iqr", "error_iqr"]:
        if candidate in df.columns:
            df = df.copy()
            df["iqr_error"] = df[candidate]
            return df
    df = df.copy()
    df["iqr_error"] = np.nan
    return df

def _ashrae_model_map():
    return {
        "ASHRAE Baseline": ashrae_baseline_label,
        "Pseudo-Steady Baseline": ashrae_baseline_label,
        "ASHRAE $\\sigma_{q_n}$ Fit": ashrae_q_label,
        "$\\sigma_{q_n}$ ASHRAE-Corrected": ashrae_q_label,
        "ASHRAE $\\sigma_{\\Delta C_p}$ Fit": ashrae_p_label,
        "$\\sigma_{\\Delta C_p}$ ASHRAE-Corrected": ashrae_p_label,
    }

def _ashrae_fit_param_lookup(ashrae_fit_params_df):
    if ashrae_fit_params_df.empty:
        return pd.DataFrame(columns=["cd", "sigma"])
    direction_map = {"Flow Entering": "+", "Flow Exiting": "-"}
    fit_params = ashrae_fit_params_df.copy()
    fit_params["table_model"] = fit_params["model_name"].map(_ashrae_model_map()).fillna(fit_params["model_name"])
    fit_params["direction_symbol"] = fit_params["direction"].map(direction_map)
    fit_params = fit_params.dropna(subset=["table_model", "direction_symbol"])
    if fit_params.empty:
        return pd.DataFrame(columns=["cd", "sigma"])
    return fit_params.set_index(["table_model", "direction_symbol"])[["cd", "sigma"]]

def _ashrae_opening_metrics_table(ashrae_error_df, ashrae_fit_params_df):
    if ashrae_error_df.empty:
        return pd.DataFrame(columns=["model_name", "table_model", "opening_group", "direction_symbol", "cd", "sigma", "rmse", "bias", "iqr_error", "nrmse", "n"])

    ashrae_error_df = _add_iqr_error_column(ashrae_error_df)
    model_map = _ashrae_model_map()
    fit_param_lookup = _ashrae_fit_param_lookup(ashrae_fit_params_df)
    direction_map = {"Flow Entering": "+", "Flow Exiting": "-"}

    opening_ashrae = (
        ashrae_error_df[
            (ashrae_error_df["dataset"] == "windows")
            & (ashrae_error_df["scope"] == "group")
            & (ashrae_error_df["model_name"].isin(model_map))
        ]
        .copy()
    )
    if opening_ashrae.empty:
        return pd.DataFrame(columns=["model_name", "table_model", "opening_group", "direction_symbol", "cd", "sigma", "rmse", "bias", "iqr_error", "nrmse", "n"])

    opening_ashrae["table_model"] = opening_ashrae["model_name"].map(model_map)
    opening_ashrae["model_name"] = opening_ashrae["table_model"]
    opening_ashrae["opening_group"] = "Window"
    opening_ashrae["direction_symbol"] = opening_ashrae["group"].map(direction_map)
    opening_ashrae["cd"] = np.nan
    opening_ashrae["sigma"] = np.nan
    if not fit_param_lookup.empty:
        lookup_index = opening_ashrae.set_index(["table_model", "direction_symbol"]).index
        opening_ashrae["cd"] = [fit_param_lookup["cd"].get(key, np.nan) for key in lookup_index]
        opening_ashrae["sigma"] = [fit_param_lookup["sigma"].get(key, np.nan) for key in lookup_index]
    opening_ashrae["opening_group"] = pd.Categorical(opening_ashrae["opening_group"], categories=["Window", "Skylight"], ordered=True)
    opening_ashrae["direction_symbol"] = pd.Categorical(opening_ashrae["direction_symbol"], categories=["+", "-"], ordered=True)
    opening_ashrae["model_order"] = pd.Categorical(opening_ashrae["table_model"], categories=original_table_model_order, ordered=True)
    return opening_ashrae[["model_name", "table_model", "opening_group", "direction_symbol", "model_order", "cd", "sigma", "rmse", "bias", "iqr_error", "nrmse", "n"]]

def _ashrae_room_metrics_table(ashrae_error_df):
    if ashrae_error_df.empty:
        return pd.DataFrame(columns=["model_name", "table_model", "opening_group", "room_type", "rmse", "bias", "iqr_error", "nrmse", "n"])

    ashrae_error_df = _add_iqr_error_column(ashrae_error_df)
    model_map = _ashrae_model_map()

    room_ashrae = (
        ashrae_error_df[
            (ashrae_error_df["dataset"] == "rooms")
            & (ashrae_error_df["scope"] == "group")
            & (ashrae_error_df["model_name"].isin(model_map))
        ]
        .copy()
    )
    if room_ashrae.empty:
        return pd.DataFrame(columns=["model_name", "table_model", "opening_group", "room_type", "rmse", "bias", "iqr_error", "nrmse", "n"])

    room_ashrae["table_model"] = room_ashrae["model_name"].map(model_map)
    room_ashrae["model_name"] = room_ashrae["table_model"]
    room_ashrae["opening_group"] = "Window"
    room_ashrae["room_type"] = room_ashrae["group"]
    room_ashrae = room_ashrae[room_ashrae["room_type"].isin(room_type_order)].copy()
    room_ashrae["opening_group"] = pd.Categorical(room_ashrae["opening_group"], categories=["Window", "Skylight"], ordered=True)
    room_ashrae["room_type"] = pd.Categorical(room_ashrae["room_type"], categories=room_type_order, ordered=True)
    room_ashrae["model_order"] = pd.Categorical(room_ashrae["table_model"], categories=original_table_model_order, ordered=True)
    return room_ashrae[["model_name", "table_model", "opening_group", "room_type", "model_order", "rmse", "bias", "iqr_error", "nrmse", "n"]]

error_metrics_tidy = _add_iqr_error_column(error_metrics_tidy)

opening_metric_values = (
    error_metrics_tidy[
        (error_metrics_tidy["source"].isin(["plot_ventilation_model_fit", "direct_network_i_type"]))
        & (error_metrics_tidy["scope"] == "subgroup")
        & (error_metrics_tidy["model_name"].isin(original_table_model_order))
    ]
    .assign(
        table_model=lambda df: df["model_name"].replace(original_table_label_map),
        opening_group=lambda df: pd.Categorical(
            df["opening_group"], categories=["Window", "Skylight"], ordered=True
        ),
        direction_symbol=lambda df: pd.Categorical(
            df["direction"].map({"Flow Entering": "+", "Flow Exiting": "-"}),
            categories=["+", "-"],
            ordered=True,
        ),
        model_order=lambda df: pd.Categorical(
            df["model_name"], categories=original_table_model_order, ordered=True
        ),
    )
)

original_table_metrics = (
    opening_metric_values
    .groupby(
        ["model_name", "table_model", "opening_group", "direction_symbol", "model_order"],
        observed=True,
        dropna=False,
        sort=False,
    )
    .agg(
        cd=("cd", "mean"),
        sigma=("sigma", "mean"),
        rmse=("rmse", "mean"),
        bias=("bias", "mean"),
        iqr_error=("iqr_error", "mean"),
        nrmse=("nrmse", "mean"),
        n=("n", "sum"),
    )
    .reset_index()
    .sort_values(["opening_group", "model_order", "direction_symbol"])
    .drop(columns=["model_order"])
)

ashrae_error_metrics_df = pd.read_csv(ashrae_error_metrics_path) if os.path.exists(ashrae_error_metrics_path) else pd.DataFrame()
ashrae_fit_params_df = pd.read_csv(ashrae_fit_params_path) if os.path.exists(ashrae_fit_params_path) else pd.DataFrame()
ashrae_opening_metrics = _ashrae_opening_metrics_table(ashrae_error_metrics_df, ashrae_fit_params_df)
if not ashrae_opening_metrics.empty:
    original_table_metrics = (
        pd.concat([original_table_metrics, ashrae_opening_metrics.drop(columns=["model_order"])], ignore_index=True, sort=False)
        .assign(model_order=lambda df: pd.Categorical(df["table_model"], categories=original_table_model_order, ordered=True))
        .sort_values(["opening_group", "model_order", "direction_symbol"])
        .drop(columns=["model_order"])
        .reset_index(drop=True)
    )

room_metric_values = (
    error_metrics_tidy[
        (error_metrics_tidy["source"].isin(["plot_ventilation_model_fit", "direct_network_i_type"]))
        & (error_metrics_tidy["scope"] == "room_slall")
        & (error_metrics_tidy["room_type"].isin(room_type_order))
        & (error_metrics_tidy["model_name"].isin(original_table_model_order))
    ]
    .assign(
        table_model=lambda df: df["model_name"].replace(original_table_label_map),
        opening_group=lambda df: pd.Categorical(
            df["opening_group"], categories=["Window", "Skylight"], ordered=True
        ),
        room_type=lambda df: pd.Categorical(
            df["room_type"], categories=room_type_order, ordered=True
        ),
        model_order=lambda df: pd.Categorical(
            df["model_name"], categories=original_table_model_order, ordered=True
        ),
    )
)

original_table_room_type_metrics = (
    room_metric_values
    .groupby(
        ["model_name", "table_model", "opening_group", "room_type", "model_order"],
        observed=True,
        dropna=False,
        sort=False,
    )
    .agg(
        rmse=("rmse", "mean"),
        bias=("bias", "mean"),
        iqr_error=("iqr_error", "mean"),
        nrmse=("nrmse", "mean"),
        n=("n", "sum"),
    )
    .reset_index()
    .sort_values(["opening_group", "room_type", "model_order"])
    .drop(columns=["model_order"])
)

ashrae_room_metrics = _ashrae_room_metrics_table(ashrae_error_metrics_df)
if not ashrae_room_metrics.empty:
    original_table_room_type_metrics = (
        pd.concat([original_table_room_type_metrics, ashrae_room_metrics.drop(columns=["model_order"])], ignore_index=True, sort=False)
        .assign(model_order=lambda df: pd.Categorical(df["table_model"], categories=original_table_model_order, ordered=True))
        .sort_values(["opening_group", "room_type", "model_order"])
        .drop(columns=["model_order"])
        .reset_index(drop=True)
    )

display(
    original_table_metrics[
        [
            "table_model",
            "opening_group",
            "direction_symbol",
            "cd",
            "sigma",
            "rmse",
            "bias",
            "iqr_error",
            "nrmse",
            "n",
        ]
    ].rename(
        columns={
            "table_model": "Model",
            "opening_group": "Opening",
            "direction_symbol": "Direction",
            "cd": "C_d",
            "rmse": "Opening RMSE Mean",
            "bias": "Opening Bias Mean",
            "iqr_error": "Opening Error IQR Mean",
            "nrmse": "Opening NRMSE Mean",
            "n": "Opening n",
        }
    )
)

display(
    original_table_room_type_metrics[
        [
            "table_model",
            "opening_group",
            "room_type",
            "rmse",
            "bias",
            "iqr_error",
            "nrmse",
            "n",
        ]
    ].rename(
        columns={
            "table_model": "Model",
            "opening_group": "Room Group",
            "room_type": "Room Type",
            "rmse": "Room-Type RMSE Mean",
            "bias": "Room-Type Bias Mean",
            "iqr_error": "Room-Type Error IQR Mean",
            "nrmse": "Room-Type NRMSE Mean",
            "n": "Room-Type n",
        }
    )
)


In [ ]:
from matplotlib.lines import Line2D

ashrae_plot_order = [ashrae_baseline_label, ashrae_q_label, ashrae_p_label]
non_ashrae_plot_order = [model for model in original_table_model_order if model not in ashrae_plot_order]
# Matplotlib draws y=0 at the bottom, so reverse the desired visual top-to-bottom order.
model_order = (ashrae_plot_order + non_ashrae_plot_order)[::-1]
metric_rows = [("rmse", "RMSE"), ("bias", "Bias")]
opening_cols = [("Window", "Windows"), ("Skylight", "Skylights")]
direction_palette = {"+": "red", "-": "blue"}


def _plot_point(ax, y_pos, value, color):
    if pd.isna(value):
        return
    ax.scatter(
        value,
        y_pos,
        color=color,
        edgecolor="black",
        linewidth=0.6,
        s=42,
        zorder=3,
    )


def _plot_connector(ax, y_pos, values):
    values = [value for value in values if pd.notna(value)]
    if len(values) < 2:
        return
    ax.plot(
        values,
        [y_pos] * len(values),
        color="0.35",
        linewidth=1.2,
        alpha=0.25,
        zorder=2,
    )


fig_opening, axs_opening = plt.subplots(
    len(opening_cols),
    len(metric_rows),
    figsize=(11.5, 9.2),
    dpi=160,
    sharex=False,
    sharey=True,
)

for row_idx, (opening, opening_label) in enumerate(opening_cols):
    for col_idx, (metric_col, metric_label) in enumerate(metric_rows):
        ax = axs_opening[row_idx, col_idx]
        opening_data = original_table_metrics[original_table_metrics["opening_group"] == opening]

        for model_idx, model in enumerate(model_order):
            point_values = []
            for direction in ["+", "-"]:
                row = opening_data[
                    (opening_data["table_model"] == model)
                    & (opening_data["direction_symbol"] == direction)
                ]
                if row.empty:
                    continue
                value = row.iloc[0][metric_col]
                point_values.append(value)
                _plot_point(
                    ax,
                    model_idx,
                    value,
                    direction_palette[direction],
                )
            _plot_connector(ax, model_idx, point_values)

        ax.axvline(0, color="k", linewidth=0.8, alpha=0.7)
        ax.grid(axis="x", alpha=0.25)
        ax.grid(axis="y", alpha=0.35, linestyle="--")
        ax.set_yticks(range(len(model_order)))
        ax.set_yticklabels(model_order, fontsize=16)
        ax.tick_params(axis="x", labelsize=16)
        ax.tick_params(axis="y", labelsize=16)
        ax.set_ylim(-0.5, len(model_order) - 0.5)
        if row_idx == 0:
            ax.set_title(metric_label, fontsize=18)
        if col_idx == 0:
            ax.set_ylabel(opening_label, fontsize=16)

flow_handles = [
    Line2D([0], [0], color=direction_palette[direction], marker="o", linestyle="None", markersize=9, label=f"{direction} opening flow")
    for direction in ["+", "-"]
]
fig_opening.legend(
    handles=flow_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=2,
    fontsize=16,
    frameon=False,
)
fig_opening.tight_layout(rect=[0, 0.06, 1, 1])

room_types = room_type_order
room_palette = dict(zip(room_types, sns.color_palette("colorblind", n_colors=len(room_types))))
slall_cols = [("Window", "Skylights Closed"), ("Skylight", "Skylights Open")]

if original_table_room_type_metrics.empty:
    raise ValueError("No room-type metrics found for the room-level RMSE/Bias plot.")

fig_room, axs_room = plt.subplots(
    len(slall_cols),
    len(metric_rows),
    figsize=(11.5, 9.2),
    dpi=160,
    sharex=False,
    sharey=True,
)

for row_idx, (opening_group, title) in enumerate(slall_cols):
    for col_idx, (metric_col, metric_label) in enumerate(metric_rows):
        ax = axs_room[row_idx, col_idx]
        slall_data = original_table_room_type_metrics[original_table_room_type_metrics["opening_group"] == opening_group]

        for model_idx, model in enumerate(model_order):
            point_values = []
            for room_type in room_types:
                row = slall_data[
                    (slall_data["table_model"] == model)
                    & (slall_data["room_type"] == room_type)
                ]
                if row.empty:
                    continue
                value = row.iloc[0][metric_col]
                point_values.append(value)
                _plot_point(
                    ax,
                    model_idx,
                    value,
                    room_palette[room_type],
                )
            _plot_connector(ax, model_idx, point_values)

        ax.axvline(0, color="k", linewidth=0.8, alpha=0.7)
        ax.grid(axis="x", alpha=0.25)
        ax.grid(axis="y", alpha=0.35, linestyle="--")
        ax.set_yticks(range(len(model_order)))
        ax.set_yticklabels(model_order, fontsize=16)
        ax.tick_params(axis="x", labelsize=16)
        ax.tick_params(axis="y", labelsize=16)
        ax.set_ylim(-0.5, len(model_order) - 0.5)
        if row_idx == 0:
            ax.set_title(metric_label, fontsize=18)
        if col_idx == 0:
            ax.set_ylabel(title, fontsize=16)

room_handles = [
    Line2D([0], [0], color=room_palette[room_type], marker="o", linestyle="None", markersize=9, label=room_type.title())
    for room_type in room_types
]
fig_room.legend(
    handles=room_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=len(room_types),
    fontsize=16,
    frameon=False,
)
fig_room.tight_layout(rect=[0, 0.06, 1, 1])
